In [1]:
import pandas as pd
from pathlib import Path
from tabulate import tabulate
from openai import OpenAI

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

df = pd.read_json(PROJECT_ROOT / "datasets" / "ViNumQA" / "test.json")
df.sample(n=5)

,pre_text,table,post_text,id,qa
7,[đánh giá quý 1: phương pháp quản trị thận trọ...,"[[Năm tài chính (31/12), 12/16, 12/17, 12/18, ...",[.],masvn/2020/2020053-200427_VCB_1Q20_review_vn/p...,{'question': 'tốc độ tăng trưởng lợi nhuận sau...
230,"[tính đến cuối 2019, ppc có 1.284 tỷ đồng nợ n...","[[, 2016, 2017, 2018, 2019], [Tỷ suất lợi nhuậ...",[.],masvn/2020/2020040-PPC_Báocáol_n__u-05022020/p...,{'question': 'tổng lượng tiền mặt và các khoản...
282,[tái cơ cấu nợ có vấn đề (tdr) tdr là một khoả...,"[[tính bằng triệu đô la, ngày 31 tháng 12 năm ...",[( a ) theo hướng dẫn của cơ quan quản lý ban ...,PNC/2012/page_174.pdf-5,"{'question': 'Trong năm 2012, tỷ lệ phần trăm ..."
383,[thỏa thuận tín dụng ngân hàng tuần hoàn đã ca...,"[[đơn vị triệu đô la, 2010, 2011, 2012, 2013, ...",[(a) tổng nợ chỉ bao gồm các khoản thanh toán ...,IP/2009/page_45.pdf-2,{'question': 'Tỷ lệ phần trăm của các nghĩa vụ...
227,[thông tin bổ sung về các hoạt động sản xuất d...,"[[(tính bằng triệu), 2004, 2003, 2002], [doanh...",[.],MRO/2004/page_125.pdf-2,{'question': 'Tỷ lệ giảm chi phí phát triển tr...


In [2]:
len(df)

497

In [3]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

df["pre_text_processed"] = df.apply(lambda x: formatting_pre_text(x), axis=1)
df["post_text_processed"] = df.apply(lambda x: formatting_post_text(x), axis=1)
df["table_processed"] = df.apply(lambda x: formatting_table(x), axis=1)
df["table_raw"] = df["table"]  # keep raw rows for table_* row-name lookup at eval time
df["input_question"] = df.apply(lambda x: processing_input_question(x), axis=1)
df["program_processed"] = df.apply(lambda x: processing_program_content(x), axis=1)
df["answer_processed"] = df.apply(lambda x: processing_answer_content(x), axis=1)
df.sample(n=5)

,pre_text,table,post_text,id,qa,pre_text_processed,post_text_processed,table_processed,table_raw,input_question,program_processed,answer_processed
47,"[năm 2019, hpg đã cung cấp gần 2,8 triệu tấn t...","[[(Tỷ đồng), FY 2015, FY 2016, FY 2017, FY 201...",[.],masvn/2020/2020056-HPG_Companynote_MAS22.05.20...,{'question': 'Tỷ lệ ROE (%) của HPG trong năm ...,"năm 2019, hpg đã cung cấp gần 2,8 triệu tấn th...",.,| (Tỷ đồng) | FY 2015 | FY 2016 |...,"[[(Tỷ đồng), FY 2015, FY 2016, FY 2017, FY 201...",Tỷ lệ ROE (%) của HPG trong năm 2020 dự kiến g...,"subtract(14.6, 17.1)",-2.5
118,[kqkd 9m19 của pme đã ghi nhận chứng kiến về t...,"[[FY (Dec.), 12/15, 12/16, 12/17, 12/18, 12/19...",[.],masvn/2020/2020036-MAS_VN_PME_Initiate_Nov_14/...,{'question': 'doanh thu cao nhất (tỷ VNĐ) được...,kqkd 9m19 của pme đã ghi nhận chứng kiến về tă...,.,| FY (Dec.) | 12/15 | 12/16 ...,"[[FY (Dec.), 12/15, 12/16, 12/17, 12/18, 12/19...",doanh thu cao nhất (tỷ VNĐ) được ghi nhận tron...,"table_max(Doanh thu (VNDbn), none)",1781.0
455,[kqkd 9m19 của pme đã ghi nhận chứng kiến về t...,"[[FY (Dec.), 12/15, 12/16, 12/17, 12/18, 12/19...",[.],masvn/2020/2020036-MAS_VN_PME_Initiate_Nov_14/...,{'question': 'Lợi nhuận sau thuế (LNST) của PM...,kqkd 9m19 của pme đã ghi nhận chứng kiến về tă...,.,| FY (Dec.) | 12/15 | 12/16 ...,"[[FY (Dec.), 12/15, 12/16, 12/17, 12/18, 12/19...",Lợi nhuận sau thuế (LNST) của PME giảm bao nhi...,"subtract(307, 309), divide(#0, 309)",-0.00647
322,[thuyết minh báo cáo tài chính hợp nhất ( tiếp...,"[[(đơn vị tính: triệu đô la), 2013, 2012], [cá...",[các khoản phải thu tài chính và hợp đồng soc ...,SNA/2013/page_84.pdf-3,"{'question': 'Trong năm 2013, bao nhiêu phần t...",thuyết minh báo cáo tài chính hợp nhất ( tiếp ...,các khoản phải thu tài chính và hợp đồng soc t...,| (đơn vị tính: triệu đô la) ...,"[[(đơn vị tính: triệu đô la), 2013, 2012], [cá...","Trong năm 2013, bao nhiêu phần trăm các khoản ...","divide(14.9, 546.5)",0.02726
126,[thuyết minh báo cáo tài chính hợp nhất giá tr...,"[[Đơn vị: triệu $, tính đến tháng 12 năm 2012,...",[1 thể hiện tác động lên các công cụ phái sinh...,GS/2012/page_121.pdf-2,{'question': 'Tỷ lệ thay đổi phần trăm trong t...,thuyết minh báo cáo tài chính hợp nhất giá trị...,1 thể hiện tác động lên các công cụ phái sinh ...,| Đơn vị: triệu $ ...,"[[Đơn vị: triệu $, tính đến tháng 12 năm 2012,...",Tỷ lệ thay đổi phần trăm trong tổng các khoản ...,"subtract(377677, 388669), divide(#0, 388669)",-0.02828


In [4]:
df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question", "program_processed", "answer_processed"]]
df.columns = [["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]]
df["generated_program"] = ""
df["calculated_program"] = ""
df

,pre_text,table,table_raw,post_text,question,program,answer,generated_program,calculated_program
0,thuyết minh báo cáo tài chính hợp nhất ( tiếp ...,| các thành phần của ảnh hưởng lũy kế của việc...,[[các thành phần của ảnh hưởng lũy kế của việc...,.,Sự thay đổi trong thu nhập ròng từ hiệu ứng tí...,"add(30, 1)",31.0,,
1,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,"Theo dự phóng, doanh thu và lợi nhuận ròng quý...","subtract(9829, 642)",9187.0,,
2,"sử dụng phương pháp p/b và rnav để định giá, c...",| Năm tài chính (31/12) | 2016 | 2017 | ...,"[[Năm tài chính (31/12), 2016, 2017, 2018, 201...",.,IDC có bao nhiêu ha quỹ đất sẵn sàng cho thuê ...,"add(495, 398)",893.0,,
3,27/10/13 26/10/14 25/10/15 30/10/16 29/10/17 2...,| | 27/10/2013 |...,"[[, 27/10/2013, 26/10/2014, 25/10/2015, 30/10/...",.,Tỷ suất lợi nhuận trên đầu tư (ROI) của Applie...,"subtract(96.67, 100), divide(#0, 100)",-0.0333,,
4,thông tin tài chính bổ sung hiệu suất cổ phiếu...,| | 12/26/08 | ...,"[[, 12/26/08, 12/31/09, 12/31/10, 12/31/11, 12...",218 báo cáo thường niên năm 2013 của goldman s...,tỷ lệ lợi nhuận tích lũy tổng cộng theo phần t...,"subtract(248.36, 100), divide(#0, 100)",1.4836,,
...,...,...,...,...,...,...,...,...,...
492,thuyết minh báo cáo tài chính hợp nhất năm 201...,| ...,"[[, 2008, 2007], [Số dư đầu kỳ, $ 134.8, $ 266...",trong tổng số lợi ích thuế chưa được ghi nhận ...,Tỷ lệ phần trăm lợi ích thuế chưa được công nh...,"divide(131.8, 148.8)",0.88575,,
493,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,Doanh thu trung bình từ năm tài chính 2017 đến...,"add(29710, 32662), add(#0, 35374), divide(#1, 3)",32582.0,,
494,định giá và khuyến nghị:\nchúng tôi khuyến ngh...,| Năm tài chính (31/12) | FY17 | FY1...,"[[Năm tài chính (31/12), FY17, FY18, FY19, FY2...",.,Tính phần trăm thay đổi EPS từ năm 2020 đến 2021.,"subtract(962, 788), divide(#0, 788)",0.22081,,
495,"trong quá trình kinh doanh thông thường, dựa t...",| ( đơn vị: nghìn ) | diện tích ròng chưa ph...,"[[( đơn vị: nghìn ), diện tích ròng chưa phát ...",( a ) một giếng khoan thăm dò được lên kế hoạc...,Tỷ lệ phần trăm diện tích đất chưa phát triển ...,"divide(145, 586)",0.24744,,


In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ["API_KEY"]
BASE_URL = os.environ["BASE_URL"]

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

MODEL = "DeepSeek-V4-Flash" # Llama-3.3-70B-Instruct, DeepSeek-V4-Flash, gemma-3-27b-it

In [6]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}
 
[TABLE]
{table}
 
[TEXT AFTER TABLE]
{post_text}
 
### QUESTION:
{question}
 
### PROGRAM:"""

# 3 few-shot demonstrations sampled from train.json, one per evidence type
# (Table Only / Text Only / Table & Text), following the ViNumQA task's
# own categorization (see VLSP 2025 NumQA paper, Section 3.3).
# The "table" field of each shot is pre-rendered to the same GitHub-markdown
# format that `formatting_table` produces for the real data, so the few-shot
# demonstrations are formatted identically to the actual queries.
FEW_SHOT_EXAMPLES = [
    {
        # Table Only (train idx 1959): answer is derived purely from two table cells.
        "pre_text": "phụ lục iv ace limited và các công ty con thông tin bổ sung về phí tái bảo hiểm thu được cho các năm kết thúc ngày 31 tháng 12 năm 2010, 2009 và 2008 (tính bằng triệu đô la mỹ, ngoại trừ tỷ lệ phần trăm) số tiền trực tiếp nhượng cho các công ty nhận từ các công ty khác số tiền ròng tỷ lệ phần trăm số tiền nhận được trên.",
        "table": (
            "|   cho các năm kết thúc ngày 31 tháng 12 năm 2010, 2009 và 2008 (tính bằng triệu đô la Mỹ, ngoại trừ tỷ lệ phần trăm) | số tiền trực tiếp   | nhượng cho các công ty khác   | nhận từ các công ty khác   | số tiền ròng   | tỷ lệ phần trăm số tiền nhận được trên số tiền ròng   |\n"
            "|----------------------------------------------------------------------------------------------------------------------|---------------------|-------------------------------|----------------------------|----------------|-------------------------------------------------------|\n"
            "|                                                                                                                 2010 | $ 15780             | $ 5792                        | $ 3516                     | $ 13504        | 26% ( 26 % )                                          |\n"
            "|                                                                                                                 2009 | $ 15415             | $ 5943                        | $ 3768                     | $ 13240        | 28% ( 28 % )                                          |\n"
            "|                                                                                                                 2008 | $ 16087             | $ 6144                        | $ 3260                     | $ 13203        | 25% ( 25 % )                                          |"
        ),
        "post_text": ".",
        "question": "Sự khác biệt giữa số tiền chuyển giao và nhận chuyển giao trong năm 2010 là bao nhiêu?",
        "program": "subtract(5792, 3516)",
    },
    {
        # Text Only (train idx 93): the supplied table (VHM financial summary) is
        # irrelevant to the question; the program only uses numbers from pre_text.
        "pre_text": "hệ số khả năng thanh toán lãi vay cũng tăng cao đạt mức 13.2 lần, so với chỉ 10.3 lần cùng kỳ.",
        "table": (
            "|                   |   FY 2015 |   FY 2016 |   FY 2017 |   FY 2018 |   FY 2019(F) |\n"
            "|-------------------|-----------|-----------|-----------|-----------|--------------|\n"
            "| Doanh thu (VNDbn) |      4920 |     11217 |     15297 |     38664 |        71115 |\n"
            "| Lãi gộp (Vbn)     |       718 |      2420 |      3128 |      7617 |        10983 |"
        ),
        "post_text": ".",
        "question": "Hệ số khả năng thanh toán lãi vay tăng bao nhiêu lần so với cùng kỳ năm ngoái?",
        "program": "subtract(13.2, 10.3)",
    },
    {
        # Table & Text (train idx 1127): must locate the right table row ("Nội dung số")
        # across three columns and chain two operators via the #0 reference.
        "pre_text": "tỷ lệ phần trăm chi phí vốn trên phần trăm tổng tài sản của mảng viễn thông được duy trì trên 1, cho thấy sự tập trung phân bổ chi phí vốn vào mảng viễn thông của fpt qua các năm.\nngoài ra, tỷ lệ này của mảng đầu tư và giáo dục là 1,1 vào năm 2018 và 0,9 vào năm 2019, khẳng định fpt cũng đang tập trung vào phát triển 2 mảng này trong 2 năm gần đây.",
        "table": (
            "|                     |   2014 |   2015 |   2016 |   2017 |   2018 |   2019 |\n"
            "|---------------------|--------|--------|--------|--------|--------|--------|\n"
            "| Viễn thông          |    1.8 |    2.4 |    1.9 |    1.4 |    1.7 |    1.8 |\n"
            "| Nội dung số         |    0.4 |    0.2 |    0.9 |    0.1 |    0.1 |    0.1 |\n"
            "| Phát triển phần mềm |    2.4 |    1.3 |    3.1 |    1.1 |    0.4 |    0.5 |"
        ),
        "post_text": ".",
        "question": "Tổng tỷ lệ của mảng Nội dung số trong ba năm từ 2014 đến 2016 là bao nhiêu?",
        "program": "add(0.4, 0.2), add(#0, 0.9)",
    },
]

In [7]:
from tqdm import tqdm

# Build the fixed few-shot prefix once: alternating user/assistant turns,
# one pair per FEW_SHOT_EXAMPLES entry, each following the same USER_MESSAGE_FRAME
# used for the real query so the model sees a consistent input/output format.
few_shot_messages = []
for shot in FEW_SHOT_EXAMPLES:
    few_shot_messages.append({
        "role": "user",
        "content": USER_MESSAGE_FRAME.format(
            pre_text=shot["pre_text"],
            table=shot["table"],
            post_text=shot["post_text"],
            question=shot["question"],
        )
    })
    few_shot_messages.append({"role": "assistant", "content": shot["program"]})

for df_index, values in tqdm(df.iterrows(), total=len(df), desc="Generating program..."):
    pre_text = values["pre_text"]
    table = values["table"]
    post_text = values["post_text"]
    question = values["question"]

    chat_completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_MESSAGE,
            },
            *few_shot_messages,
            {
                "role": "user",
                "content": USER_MESSAGE_FRAME.format(pre_text=pre_text, table=table, post_text=post_text, question=question)
            },
        ],
            temperature=0.0,
            max_tokens=8096,
            stream=False
            )
    output = chat_completion.choices[0].message.content.strip().strip("\n")

    df.at[df_index, "generated_program"] = output

    print(f"TEST SAMPLE {df_index}:\n\nPREDICTION:\n{output}\n\nGROUND_TRUTH:\n{values['program']}")
    print("="*100)

Generating program...:   0%|          | 1/497 [00:01<14:54,  1.80s/it]

TEST SAMPLE 0:

PREDICTION:
add(-54, 30), add(#0, 1)

GROUND_TRUTH:
add(30, 1)


Generating program...:   0%|          | 2/497 [00:03<12:05,  1.47s/it]

TEST SAMPLE 1:

PREDICTION:
subtract(9829, 642)

GROUND_TRUTH:
subtract(9829, 642)


Generating program...:   1%|          | 3/497 [00:07<23:26,  2.85s/it]

...

TEST SAMPLE 494:

PREDICTION:
subtract(962, 788), divide(#0, 788), multiply(#1, 100)

GROUND_TRUTH:
subtract(962, 788), divide(#0, 788)


Generating program...: 100%|█████████▉| 496/497 [19:09<00:01,  1.59s/it]

TEST SAMPLE 495:

PREDICTION:
divide(145, 586), multiply(#0, 100)

GROUND_TRUTH:
divide(145, 586)


Generating program...: 100%|██████████| 497/497 [19:11<00:00,  2.32s/it]

TEST SAMPLE 496:

PREDICTION:
multiply(1041, 100), divide(#0, 29.2)

GROUND_TRUTH:
divide(1041, 0.292)


In [8]:
import sys
sys.path.insert(0, "../../evaluate")  # fallback: relative path when running locally

from scorer import evaluate_dataframe  # noqa: E402

# scorer.py is the shared ViNumQA evaluator (notebooks/evaluate/scorer.py): it
# ports FinQA's official evaluation protocol (sympy-based symbolic Program
# Accuracy, table-row-lookup-aware Execution Accuracy) instead of a
# hand-rolled parser, and correctly executes table_*(row_name, none) calls by
# looking up the named row in the raw table -- which the previous in-notebook
# parser could not do at all (it treated table_* arguments as raw numbers).

In [9]:
df_scored, summary = evaluate_dataframe(
    df,
    generated_col="generated_program",   # cột bạn đang ghi output model vào
    gold_program_col="program",
    gold_answer_col="answer",
    table_col="table_raw",
)

print(summary)  # {'program_accuracy': ..., 'execution_accuracy': ...}

{'program_accuracy': 0.44668008048289737, 'execution_accuracy': 0.5050301810865191}
